# 🌲 Thực Hành: Huấn Luyện & Tinh Chỉnh Siêu Tham Số XGBoost & LightGBM

Chào Khang! Trong bài tập thực hành này, bạn sẽ làm việc với một tập dữ liệu giả lập về **Dự Đoán Khách Hàng Rời Bỏ Dịch Vụ (Churn Prediction)** của một ngân hàng thương mại. Đây là một bài toán phân loại nhị phân dữ liệu bảng cực kỳ phổ biến trong các doanh nghiệp ngày nay.

Bạn sẽ học cách:
1. Huấn luyện các mô hình **XGBoost** và **LightGBM** cơ bản.
2. Thực hiện kỹ thuật tinh chỉnh siêu tham số (**Hyperparameter Tuning**) để chống Overfitting.
3. Trích xuất và vẽ biểu đồ **Feature Importance** để đưa ra lý giải kinh doanh.

Trước khi bắt đầu, hãy đảm bảo bạn đã cài đặt hai thư viện này bằng cách chạy ô code bên dưới.

In [ ]:
!pip install xgboost lightgbm scikit-learn matplotlib pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

# Thiết lập hạt giống ngẫu nhiên
np.random.seed(42)
print("Đã tải thành công các thư viện cần thiết!")

---
## 1. Chuẩn Bị Dữ Liệu Bảng (Tabular Churn Dataset)

Chúng ta sẽ tự tạo một tập dữ liệu dạng bảng giả lập biểu thị thông tin giao dịch của **2000 khách hàng** với các đặc trưng:
- `Age` (Tuổi)
- `Balance` (Số dư tài khoản)
- `NumOfProducts` (Số lượng sản phẩm sử dụng)
- `HasCrCard` (Có thẻ tín dụng hay không)
- `IsActiveMember` (Có phải là thành viên tích cực hay không)
- `EstimatedSalary` (Mức lương ước tính)

Nhãn dự đoán: `Exited` (1 nếu khách hàng rời bỏ ngân hàng, 0 nếu ở lại).

In [ ]:
# Tạo dữ liệu giả lập phân loại nhị phân gồm 2000 mẫu, 6 đặc trưng
X_raw, y_raw = make_classification(
    n_samples=2000,
    n_features=6,
    n_informative=4,
    n_redundant=2,
    random_state=42
)

# Đổi tên cột cho thực tế
feature_names = ['Age', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
df = pd.DataFrame(X_raw, columns=feature_names)
df['Exited'] = y_raw

# Hiển thị 5 dòng dữ liệu đầu tiên
df.head()

In [ ]:
# Phân chia tập dữ liệu thành Train (80%) và Test (20%)
X = df[feature_names]
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

---
## 2. Huấn Luyện Mô Hình XGBoost Cơ Bản

**Nhiệm vụ của bạn:** Khởi tạo mô hình `xgb.XGBClassifier` với các tham số mặc định, huấn luyện trên tập Train (`X_train`, `y_train`), thực hiện dự đoán lớp trên tập Test và in báo cáo kết quả.

*Gợi ý:* 
- Sử dụng hàm `xgb.XGBClassifier(random_state=42)`
- Gọi `.fit(X_train, y_train)`
- Gọi `.predict(X_test)`
- Sử dụng `classification_report(y_test, y_pred)` để đánh giá.

In [ ]:
# ------------------ YOUR CODE HERE ------------------
# 1. Khởi tạo mô hình
xgb_clf = None

# 2. Huấn luyện mô hình


# 3. Dự đoán nhãn trên tập kiểm thử (Test Set)
y_pred_xgb = None
# ----------------------------------------------------

# Đánh giá kết quả
if y_pred_xgb is not None:
    print("--- KẾT QUẢ PHÂN LOẠI XGBOOST MẶC ĐỊNH ---")
    print(classification_report(y_test, y_pred_xgb))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1]):.4f}")

---
## 3. Huấn Luyện Mô Hình LightGBM Cơ Bản

**Nhiệm vụ của bạn:** Khởi tạo và huấn luyện mô hình `lgb.LGBMClassifier` tương tự như cách làm với XGBoost ở trên.

In [ ]:
# ------------------ YOUR CODE HERE ------------------
# 1. Khởi tạo mô hình LGBMClassifier
lgb_clf = None

# 2. Huấn luyện mô hình


# 3. Dự đoán nhãn trên tập kiểm thử (Test Set)
y_pred_lgb = None
# ----------------------------------------------------

# Đánh giá kết quả
if y_pred_lgb is not None:
    print("--- KẾT QUẢ PHÂN LOẠI LIGHTGBM MẶC ĐỊNH ---")
    print(classification_report(y_test, y_pred_lgb))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, lgb_clf.predict_proba(X_test)[:, 1]):.4f}")

---
## 4. Tinh Chỉnh Siêu Tham Số (Hyperparameter Tuning) Để Tránh Quá Khớp

Các mô hình Boosting mặc định thường chia cây rất sâu và học rất nhanh nên dễ bị Overfitting trên các tập dữ liệu nhỏ.

**Nhiệm vụ của bạn:** Hãy cấu hình lại mô hình XGBoost với các tham số khắt khe hơn giúp mô hình tổng quát hóa tốt hơn:
- Giảm tốc độ học xuống thấp: `learning_rate=0.03`
- Giới hạn chiều sâu của cây quyết định: `max_depth=4` (Tránh cây quá sâu phân nhánh vụn vặt)
- Bổ sung kỹ thuật trích xuất mẫu đặc trưng và dòng dữ liệu: `subsample=0.8` và `colsample_bytree=0.8`
- Tăng số lượng cây lặp lên: `n_estimators=300`

In [ ]:
# ------------------ YOUR CODE HERE ------------------
# Khởi tạo XGBoost Classifier với các siêu tham số tối ưu chống overfitting
xgb_tuned = None

# Huấn luyện mô hình


# Dự đoán kết quả
y_pred_tuned = None
# ----------------------------------------------------

# Đánh giá kết quả sau tinh chỉnh
if y_pred_tuned is not None:
    print("--- KẾT QUẢ PHÂN LOẠI XGBOOST SAU TINH CHỈNH ---")
    print(classification_report(y_test, y_pred_tuned))
    print(f"ROC-AUC Score sau tinh chỉnh: {roc_auc_score(y_test, xgb_tuned.predict_proba(X_test)[:, 1]):.4f}")

---
## 5. Trích Xuất Độ Quan Trọng Của Đặc Trưng (Feature Importance)

Một trong những điểm mạnh lớn nhất của thuật toán dựa trên cây là khả năng cung cấp tính minh bạch cao thông qua thang đo độ quan trọng của đặc trưng.

**Nhiệm vụ của bạn:** Lấy thuộc tính `.feature_importances_` từ mô hình XGBoost đã huấn luyện ở trên, gán vào DataFrame và vẽ đồ thị thanh ngang (horizontal bar chart) để thể hiện các tính năng chi phối nhất.

In [ ]:
if 'xgb_tuned' in locals() and xgb_tuned is not None:
    # ------------------ YOUR CODE HERE ------------------
    # 1. Trích xuất độ quan trọng của đặc trưng từ xgb_tuned
    importances = None
    
    # 2. Tạo DataFrame chứa tên đặc trưng và độ quan trọng
    feat_imp_df = None
    
    # 3. Sắp xếp DataFrame theo thứ tự giảm dần của độ quan trọng

    # ----------------------------------------------------
    
    # Vẽ biểu đồ thanh ngang
    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='emerald' if 'emerald' in plt.colormaps else 'teal')
    plt.gca().invert_yaxis()  # Đảo ngược trục Y để thuộc tính quan trọng nhất lên đầu
    plt.title('Độ Quan Trọng Của Các Đặc Trưng Trong Việc Dự Đoán Churn')
    plt.xlabel('Tỷ Lệ Chi Phối (Importance Score)')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

🎉 **Tuyệt vời!** Bạn đã thực hành thành công toàn bộ vòng đời huấn luyện và tối ưu hóa XGBoost & LightGBM trên dữ liệu bảng. 

### 💡 Câu hỏi đào sâu suy nghĩ cho Khang:
1. So sánh tốc độ chạy huấn luyện của XGBoost và LightGBM: Thuật toán nào chạy nhanh hơn? Tại sao?
2. Trong Hồi quy Logistic hay mạng Neural, việc chuẩn hóa dữ liệu (`StandardScaler`) là bắt buộc để Gradient Descent hội tụ tốt. Tại sao đối với XGBoost và LightGBM (thuật toán dựa trên cây), việc phân chia nút chia nhánh không bị ảnh hưởng bởi thang đo và không bắt buộc phải chuẩn hóa dữ liệu?